# P-ML5 — Extended Dataset for Regime-Specific Models

**Motivation (F10):** `RegimeEnsemble` (P-ML4) fails because the bull model only gets
128–136 training bars per fold on 3yr data — not enough to learn trend continuation.
Bull model IC stays negative (−0.044 to −0.138); P-ML4 equity (−2.5%, Sharpe +0.227)
falls below P-ML3 Exp-C (+8.8%, Sharpe +0.280).

**Hypothesis for P-ML5:** Extending the dataset to **6yr (2019–2025)** gives ~600+
bull training bars per later fold (including the 2019 mid-year surge and the 2020–21
bull run), which should allow the bull model to actually learn trend-following features.

**Architecture:** Same `RegimeEnsemble` as P-ML4 — only the training window grows.

## §1 — Config

In [1]:
import sys
from pathlib import Path

repo_root = Path("__file__").resolve().parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    "figure.dpi":        120,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         9,
})

SYMBOL        = "BTC/USDT"
SINCE         = "2019-01-01"   # Extended from 2022 (3yr → 6yr)
UNTIL         = "2025-01-01"
HORIZON       = 1              # 1d forward log-return (same as P-ML4)
N_SPLITS      = 5
TRAIN_FRAC    = 0.6
PURGE         = 1
LONG_MA       = 200
ADX_THRESH    = 25.0
MIN_BULL_BARS = 30

FEATURES = [
    "bar_ret", "bb_zscore", "rsi", "macd_hist_norm", "atr_pct",
    "bb_width", "upper_wick", "lower_wick", "hl_range",
    "vol_log_chg", "di_diff", "adx",
]

# Static reference values from prior experiments (P-ML3/P-ML4)
P_ML3_EXPC = {"return": 0.088, "sharpe": 0.280, "maxdd": -0.498}
P_ML4      = {"return": -0.025, "sharpe": 0.227, "maxdd": -0.573}
BUY_HOLD   = {"return": 2.996, "sharpe": 1.379, "maxdd": -0.354}

print(f"Dataset:       {SINCE} → {UNTIL} | 1d | horizon={HORIZON}")
print(f"Walk-forward:  {N_SPLITS} folds, train_frac={TRAIN_FRAC}, purge={PURGE}")
print(f"Regime:        SMA({LONG_MA}) + ADX>{ADX_THRESH}")
print(f"MIN_BULL_BARS: {MIN_BULL_BARS}")

Dataset:       2019-01-01 → 2025-01-01 | 1d | horizon=1
Walk-forward:  5 folds, train_frac=0.6, purge=1
Regime:        SMA(200) + ADX>25.0
MIN_BULL_BARS: 30


## §2 — Data Loading & Regime Distribution

The existing 1d cache starts from 2022. To get 2019–2021 data, we use a cache-guard:
if the earliest cached bar is after 2020-01-01, re-fetch the full range from the exchange.

In [2]:
from data.fetch import fetch_ohlcv
from ml.features import build_feature_matrix
from ml.labels import forward_return
from ml.regime import RegimeClassifier
from ml.validation import purged_wf_splits

# Cache-guard: ensure we have 2019+ data
df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d", since=SINCE, until=UNTIL)
if df_raw.index.min() > pd.Timestamp("2020-01-01", tz="UTC"):
    # Cache is missing early data — re-fetch the full range fresh
    print("Cache missing early data — re-fetching 2019–2025 from exchange...")
    df_raw = fetch_ohlcv(symbol=SYMBOL, timeframe="1d",
                         since=SINCE, until=UNTIL, use_cache=False)

print(f"Loaded {len(df_raw):,} bars  |  {df_raw.index[0].date()} → {df_raw.index[-1].date()}")

Cache missing early data — re-fetching 2019–2025 from exchange...


Loaded 2,193 bars  |  2019-01-01 → 2025-01-01


In [3]:
# Build features, labels, regime
feats = build_feature_matrix(df_raw)
label = forward_return(df_raw, horizon=HORIZON)
comb  = pd.concat([feats, label], axis=1).dropna()
X_all = comb[feats.columns]
y_all = comb[label.name]
X     = X_all[FEATURES]

rc  = RegimeClassifier(long_ma=LONG_MA, adx_thresh=ADX_THRESH)
reg = rc.transform(df_raw).reindex(X.index)
reg["regime"] = reg["regime"].fillna("ranging")
for col in ["regime_bull", "regime_bear", "regime_ranging"]:
    reg[col] = reg[col].fillna(0).astype(int)

bar_ret_daily = np.log(df_raw["close"] / df_raw["close"].shift(1)).reindex(X.index)
splits = list(purged_wf_splits(len(X), N_SPLITS, TRAIN_FRAC, purge_bars=PURGE))

print(f"{len(X):,} usable bars | {X.index[0].date()} → {X.index[-1].date()}")
print(f"\nOverall regime (6yr): bull={reg['regime_bull'].mean()*100:.1f}%  "
      f"bear={reg['regime_bear'].mean()*100:.1f}%  "
      f"ranging={reg['regime_ranging'].mean()*100:.1f}%")

# 3yr comparison (P-ML4 reported: bull=31.9%, bear=17.6%, ranging=50.5%)
print("\nRegime distribution comparison:")
print(f"  P-ML4 (3yr 2022–2025): bull=31.9%  bear=17.6%  ranging=50.5%  [1,075 bars]")
print(f"  P-ML5 (6yr 2019–2025): bull={reg['regime_bull'].mean()*100:.1f}%  "
      f"bear={reg['regime_bear'].mean()*100:.1f}%  "
      f"ranging={reg['regime_ranging'].mean()*100:.1f}%  [{len(X):,} bars]")

2,171 usable bars | 2019-01-22 → 2024-12-31

Overall regime (6yr): bull=31.6%  bear=25.6%  ranging=42.8%

Regime distribution comparison:
  P-ML4 (3yr 2022–2025): bull=31.9%  bear=17.6%  ranging=50.5%  [1,075 bars]
  P-ML5 (6yr 2019–2025): bull=31.6%  bear=25.6%  ranging=42.8%  [2,171 bars]


In [4]:
# BTC price chart with regime colouring — full 6yr span
fig, (ax_price, ax_reg) = plt.subplots(2, 1, figsize=(14, 6),
                                        gridspec_kw={"height_ratios": [4, 1]},
                                        sharex=True)

ax_price.plot(df_raw.index, df_raw["close"], color="black", linewidth=0.8, label="BTC/USDT")

# Shade regime regions
regime_colors = {"bull": "#2ecc71", "bear": "#e74c3c", "ranging": "#95a5a6"}
prev_regime   = None
seg_start     = None
for ts, row in reg.iterrows():
    r = row["regime"]
    if r != prev_regime:
        if prev_regime is not None:
            ax_price.axvspan(seg_start, ts, alpha=0.15, color=regime_colors[prev_regime], linewidth=0)
        seg_start  = ts
        prev_regime = r
if prev_regime is not None and seg_start is not None:
    ax_price.axvspan(seg_start, reg.index[-1], alpha=0.15,
                     color=regime_colors[prev_regime], linewidth=0)

patches = [mpatches.Patch(color=c, alpha=0.5, label=l)
           for l, c in regime_colors.items()]
ax_price.legend(handles=patches, fontsize=8, loc="upper left")
ax_price.set_ylabel("BTC/USDT close (USD)")
ax_price.set_title("BTC/USDT 2019–2025 with Regime Classification (SMA200 + ADX>25)",
                   fontsize=10)
ax_price.set_yscale("log")

# Regime bar below
regime_num = reg["regime"].map({"bull": 1, "ranging": 0, "bear": -1}).reindex(X.index)
ax_reg.fill_between(regime_num.index, regime_num.values,
                    step="mid", alpha=0.7,
                    color=[regime_colors[r] for r in reg["regime"]])
ax_reg.set_yticks([-1, 0, 1])
ax_reg.set_yticklabels(["bear", "ranging", "bull"], fontsize=7)
ax_reg.set_ylabel("Regime")
ax_reg.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.show()

## §3 — Walk-Forward Fold Structure

Key metric: bull training bar counts per fold. P-ML4 had 0–136 bull training bars;
P-ML5 should have substantially more per fold due to the 2019 and 2020–21 bull runs.

In [5]:
# P-ML4 reference bull bar counts (from F10)
p4_bull_bars = [0, 1, 128, 133, 136]

print(f"Walk-forward fold structure — P-ML5 (6yr, {len(X):,} bars)")
print(f"MIN_BULL_BARS = {MIN_BULL_BARS}")
print()
print(f"{'Fold':<6} {'Train period':<36} {'bull':>6} {'bear':>6} "
      f"{'ranging':>8} {'total':>7} {'bull_model?':>12} {'P-ML4 bull':>11}")
print("-" * 100)

train_bull_counts = []
for i, (tr, te) in enumerate(splits):
    vc       = reg["regime"].iloc[tr].value_counts()
    n_bull   = vc.get("bull", 0)
    has_bull = n_bull >= MIN_BULL_BARS
    train_bull_counts.append(n_bull)
    period = f"{X.index[tr[0]].date()} → {X.index[tr[-1]].date()}"
    p4_ref = p4_bull_bars[i] if i < len(p4_bull_bars) else "N/A"
    print(f"  {i+1:<4} {period:<36} "
          f"{n_bull:>6} {vc.get('bear',0):>6} {vc.get('ranging',0):>8} "
          f"{len(tr):>7} {'YES' if has_bull else 'FALLBACK':>12} {str(p4_ref):>11}")

print()
print(f"P-ML5 total bull training bars across all folds: {sum(train_bull_counts):,}")
print(f"P-ML4 total bull training bars across all folds: {sum(p4_bull_bars):,}")
print(f"Bull bar multiplier: {sum(train_bull_counts)/max(sum(p4_bull_bars),1):.1f}x")

Walk-forward fold structure — P-ML5 (6yr, 2,171 bars)
MIN_BULL_BARS = 30

Fold   Train period                           bull   bear  ranging   total  bull_model?  P-ML4 bull
----------------------------------------------------------------------------------------------------
  1    2019-01-22 → 2020-01-16                  15    192      153     360     FALLBACK           0
  2    2019-07-22 → 2021-01-11                 199    102      239     540          YES           1
  3    2020-07-17 → 2022-01-07                 273    110      157     540          YES         128
  4    2021-07-13 → 2023-01-03                  86    216      238     540          YES         133
  5    2022-07-09 → 2023-12-30                 214     82      244     540          YES         136

P-ML5 total bull training bars across all folds: 787
P-ML4 total bull training bars across all folds: 398
Bull bar multiplier: 2.0x


In [6]:
# Side-by-side bar chart: P-ML4 vs P-ML5 bull bars per fold
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_SPLITS)
w = 0.35

p4_safe = [p4_bull_bars[i] if i < len(p4_bull_bars) else 0 for i in range(N_SPLITS)]
colors_p5 = ["#2ecc71" if n >= MIN_BULL_BARS else "#e74c3c" for n in train_bull_counts]

ax.bar(x - w/2, p4_safe,           w, color="#95a5a6", alpha=0.8, label="P-ML4 (3yr)")
ax.bar(x + w/2, train_bull_counts,  w, color=colors_p5,  alpha=0.85, label="P-ML5 (6yr)")
ax.axhline(MIN_BULL_BARS, color="black", linewidth=1.0, linestyle="--",
           label=f"MIN_BULL_BARS = {MIN_BULL_BARS}")
ax.set_xticks(x)
ax.set_xticklabels([f"Fold {i+1}" for i in range(N_SPLITS)])
ax.set_ylabel("Bull training bars")
ax.set_title("Bull training bars per fold: P-ML4 (3yr) vs P-ML5 (6yr)", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## §4 — Walk-Forward Validation with RegimeEnsemble

Same logic as P-ML4: per fold, fit `RegimeEnsemble`, predict OOS, compute IC and hit-rate.

In [7]:
from ml.models import RegimeEnsemble

fold_results_P5 = []

for i, (tr, te) in enumerate(splits):
    ensemble = RegimeEnsemble(min_bull_bars=MIN_BULL_BARS)
    ensemble.fit(X.iloc[tr], y_all.iloc[tr], reg["regime"].iloc[tr])
    preds  = ensemble.predict(X.iloc[te], reg["regime"].iloc[te])
    actual = y_all.iloc[te].values

    rho, pval = stats.spearmanr(preds, actual)
    hit       = (np.sign(preds) == np.sign(actual)).mean()

    # Count bull test bars
    n_bull_train = int((reg["regime"].iloc[tr] == "bull").sum())
    n_bull_test  = int((reg["regime"].iloc[te] == "bull").sum())

    fold_results_P5.append({
        "fold":         i + 1,
        "te":           te,
        "IC":           rho,
        "IC_pval":      pval,
        "hit":          hit,
        "preds":        preds,
        "ensemble":     ensemble,
        "n_bull_train": n_bull_train,
        "n_bull_test":  n_bull_test,
        "bull_used":    ensemble.has_bull_model,
    })
    period = f"{X.index[te[0]].date()} → {X.index[te[-1]].date()}"
    print(f"Fold {i+1} [{period}]  "
          f"bull_train={n_bull_train:<4}  bull_test={n_bull_test:<4}  "
          f"bull_model={'YES' if ensemble.has_bull_model else 'NO(flip)'}  "
          f"IC={rho:+.4f}  hit={hit:.3f}")

ics_P5 = [r["IC"] for r in fold_results_P5]
print(f"\nP-ML5 ensemble: Mean IC={np.mean(ics_P5):+.4f}  "
      f"ICIR={np.mean(ics_P5)/np.std(ics_P5):.3f}")

# Compare to P-ML4
p4_ics = [-0.051, -0.053, -0.148, -0.056, -0.003]   # from F10
print(f"P-ML4 reference: Mean IC={np.mean(p4_ics):+.4f}  "
      f"ICIR={np.mean(p4_ics)/np.std(p4_ics):.3f}")

Fold 1 [2020-01-18 → 2021-01-12]  bull_train=15    bull_test=185   bull_model=NO(flip)  IC=+0.0612  hit=0.546
Fold 2 [2021-01-13 → 2022-01-08]  bull_train=199   bull_test=143   bull_model=YES  IC=-0.0536  hit=0.460
Fold 3 [2022-01-09 → 2023-01-04]  bull_train=273   bull_test=0     bull_model=YES  IC=+0.1295  hit=0.510


Fold 4 [2023-01-05 → 2023-12-31]  bull_train=86    bull_test=214   bull_model=YES  IC=+0.0462  hit=0.507
Fold 5 [2024-01-01 → 2024-12-26]  bull_train=214   bull_test=124   bull_model=YES  IC=+0.0861  hit=0.537

P-ML5 ensemble: Mean IC=+0.0539  ICIR=0.888
P-ML4 reference: Mean IC=-0.0622  ICIR=-1.319


In [8]:
# IC comparison chart: P-ML4 vs P-ML5
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(N_SPLITS)
w = 0.35
p4_ics_safe = p4_ics[:N_SPLITS]

ax.bar(x - w/2, p4_ics_safe, w, label="P-ML4 (3yr)", color="#95a5a6", alpha=0.85)
ax.bar(x + w/2, ics_P5,      w, label="P-ML5 (6yr)", color="#3498db", alpha=0.85)
ax.axhline(0,     color="black", linewidth=0.7)
ax.axhline(+0.03, color="gray", linewidth=0.6, linestyle="--", label="±0.03 target")
ax.axhline(-0.03, color="gray", linewidth=0.6, linestyle="--")
ax.set_xticks(x)
ax.set_xticklabels([f"F{i+1}" for i in range(N_SPLITS)])
ax.set_ylabel("OOS IC")
ax.set_title("OOS IC per fold: P-ML4 (3yr) vs P-ML5 (6yr) RegimeEnsemble", fontsize=10)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## §5 — Bull Model Deep-Dive

For folds where the bull model was fitted, we examine:
- Bull model OOS IC on bull test bars vs non-bull model IC on non-bull test bars
- Feature importances: bull vs non-bull (same analysis as P-ML4 §7)
- Did the bull model IC turn positive with more training data?

In [9]:
# Per-fold: bull IC vs non-bull IC
print(f"{'Fold':<6} {'Period':<30} {'n_bull_tr':>10} {'n_bull_te':>10} "
      f"{'Bull IC':>9} {'NB IC':>9} {'Bull>0?':>8}")
print("-" * 88)

bull_ics_detail = []
nb_ics_detail   = []

for r in fold_results_P5:
    i  = r["fold"] - 1
    tr, te = splits[i]
    e      = r["ensemble"]
    period = f"{X.index[te[0]].date()} → {X.index[te[-1]].date()}"

    # Non-bull IC on non-bull test bars
    nb_te_mask = (reg["regime"].iloc[te] != "bull").values
    if nb_te_mask.sum() >= 2:
        preds_nb  = e.non_bull_model.predict(X.iloc[te][nb_te_mask])
        actual_nb = y_all.iloc[te][nb_te_mask].values
        nb_ic, _  = stats.spearmanr(preds_nb, actual_nb)
    else:
        nb_ic = float("nan")
    nb_ics_detail.append(nb_ic)

    # Bull IC on bull test bars
    bull_te_mask = (reg["regime"].iloc[te] == "bull").values
    if bull_te_mask.sum() >= 2:
        if e.has_bull_model:
            preds_b   = e.bull_model.predict(X.iloc[te][bull_te_mask])
        else:
            preds_b   = -e.non_bull_model.predict(X.iloc[te][bull_te_mask])
        actual_b   = y_all.iloc[te][bull_te_mask].values
        bull_ic, _ = stats.spearmanr(preds_b, actual_b)
    else:
        bull_ic = float("nan")
    bull_ics_detail.append(bull_ic)

    method_flag = "YES" if e.has_bull_model else "FLIP"
    bull_pos    = "YES" if not np.isnan(bull_ic) and bull_ic > 0 else "no"
    print(f"  {r['fold']:<4} {period:<30} {r['n_bull_train']:>10} {r['n_bull_test']:>10} "
          f"{bull_ic:>+9.4f} {nb_ic:>+9.4f} {bull_pos:>8}  [{method_flag}]")

valid_bull = [ic for ic in bull_ics_detail if not np.isnan(ic)]
print(f"\nBull model  — Mean IC={np.nanmean(bull_ics_detail):+.4f}  "
      f"Positive in {sum(1 for ic in valid_bull if ic > 0)}/{len(valid_bull)} folds")
print(f"Non-bull    — Mean IC={np.nanmean(nb_ics_detail):+.4f}")

# P-ML4 reference bull ICs: [+1.0 (degenerate), -0.162, -0.138, -0.063, -0.044]
p4_bull_ics = [float("nan"), -0.162, -0.138, -0.063, -0.044]  # Fold1 degenerate
print(f"\nP-ML4 bull model — Mean IC={np.nanmean(p4_bull_ics):+.4f}  "
      f"(reference, negative in all fitted folds)")

Fold   Period                          n_bull_tr  n_bull_te   Bull IC     NB IC  Bull>0?
----------------------------------------------------------------------------------------
  1    2020-01-18 → 2021-01-12                15        185   +0.0312   +0.1576      YES  [FLIP]
  2    2021-01-13 → 2022-01-08               199        143   -0.0497   -0.0531       no  [YES]


  3    2022-01-09 → 2023-01-04               273          0      +nan   +0.1295       no  [YES]
  4    2023-01-05 → 2023-12-31                86        214   +0.0421   +0.0631      YES  [YES]
  5    2024-01-01 → 2024-12-26               214        124   +0.0611   +0.0892      YES  [YES]

Bull model  — Mean IC=+0.0212  Positive in 3/4 folds
Non-bull    — Mean IC=+0.0773

P-ML4 bull model — Mean IC=-0.1018  (reference, negative in all fitted folds)


In [10]:
# Feature importances: bull vs non-bull
bull_fi_list = []
nb_fi_list   = []

for r in fold_results_P5:
    e = r["ensemble"]
    nb_fi_list.append(e.non_bull_importance)
    if e.has_bull_model:
        bull_fi_list.append(e.bull_importance)

fi_nb_mean = pd.DataFrame(nb_fi_list).mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Non-bull importance
axes[0].barh(fi_nb_mean.index[::-1], fi_nb_mean.values[::-1],
             color="steelblue", alpha=0.85)
axes[0].set_title(f"Non-bull model — mean gain importance ({len(nb_fi_list)} folds)",
                  fontsize=9)
axes[0].set_xlabel("Mean gain")

# Bull importance
if bull_fi_list:
    fi_bull_mean = pd.DataFrame(bull_fi_list).mean().reindex(fi_nb_mean.index)
    axes[1].barh(fi_bull_mean.index[::-1], fi_bull_mean.values[::-1],
                 color="#2ecc71", alpha=0.85)
    axes[1].set_title(
        f"Bull model — mean gain importance ({len(bull_fi_list)} folds)", fontsize=9)
    axes[1].set_xlabel("Mean gain")

    # Print side-by-side table
    fi_cmp = pd.DataFrame({
        "non_bull": fi_nb_mean,
        "bull":     fi_bull_mean,
    }).sort_values("non_bull", ascending=False)
    plt.suptitle("Feature importance: non-bull vs bull model (P-ML5)", fontsize=10)
    plt.tight_layout()
    plt.show()
    print("\nMean gain importance — non-bull vs bull:")
    print(fi_cmp.to_string(float_format="{:.1f}".format))
else:
    axes[1].text(0.5, 0.5, "No bull models fitted",
                 ha="center", va="center", transform=axes[1].transAxes)
    plt.suptitle("Feature importance: non-bull vs bull model (P-ML5)", fontsize=10)
    plt.tight_layout()
    plt.show()


Mean gain importance — non-bull vs bull:
                non_bull  bull
upper_wick         216.8 114.2
atr_pct            205.4  88.8
vol_log_chg        195.6 136.2
bb_width           193.2 126.5
macd_hist_norm     188.4  84.5
hl_range           183.2 127.2
lower_wick         180.0 108.0
bar_ret            174.4 138.5
adx                157.6  98.5
di_diff            154.4 149.8
bb_zscore          149.0  92.8
rsi                134.0  77.8


## §6 — Equity Curve: P-ML5 vs Baselines

Stitch OOS predictions into full equity curve and compare against:
- Buy & Hold
- P-ML3 Exp-C (best prior: +8.8%, Sharpe +0.280)
- P-ML4 RegimeEnsemble (3yr: −2.5%, Sharpe +0.227)
- **P-ML5 RegimeEnsemble (6yr: computed below)**

In [11]:
from backtesting import compute_metrics

def build_equity(fold_preds_list, bar_ret_daily, X):
    """Stitch per-fold equity from a list of (te_idx, preds) tuples."""
    pieces, anchor = [], 1.0
    for te, preds in fold_preds_list:
        pos   = np.sign(preds)
        ret   = bar_ret_daily.iloc[te].values
        eq    = np.cumprod(1 + np.roll(pos, 1) * ret)
        eq[0] = 1.0
        s = pd.Series(eq, index=X.index[te])
        s = s / s.iloc[0] * anchor
        anchor = float(s.iloc[-1])
        pieces.append(s)
    return pd.concat(pieces)


# P-ML5 equity
p5_pairs = [(r["te"], r["preds"]) for r in fold_results_P5]
oos_P5   = build_equity(p5_pairs, bar_ret_daily, X)

# Buy-and-Hold (anchored to P-ML5 OOS start)
bah = df_raw["close"].reindex(oos_P5.index)
bah = bah / bah.iloc[0]

# Compute metrics
m_p5  = compute_metrics(oos_P5)
m_bah = compute_metrics(bah)

# Equity plot
fig, ax = plt.subplots(figsize=(14, 5))

oos_P5.plot(ax=ax, label="P-ML5 (6yr RegimeEnsemble)", color="#3498db", linewidth=2.0)
bah.plot(   ax=ax, label="Buy-and-Hold",                color="black",   linewidth=1.0,
            linestyle=":")

# Mark fold boundaries
for _, (tr, te) in enumerate(splits):
    ax.axvline(X.index[te[0]], color="lightgray", linewidth=0.5, linestyle="--")
ax.axhline(1, color="black", linewidth=0.4)
ax.set_title("OOS Equity — P-ML5 (6yr RegimeEnsemble) vs Buy-and-Hold", fontsize=11)
ax.set_ylabel("Equity (normalised)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print("\nEquity metrics — P-ML5 vs prior baselines:")
print(f"{'Strategy':<35} {'Return':>9} {'Sharpe':>8} {'MaxDD':>8}")
print("-" * 63)
rows = [
    ("Buy & Hold",                  BUY_HOLD["return"],     BUY_HOLD["sharpe"],     BUY_HOLD["maxdd"]),
    ("P-ML3 Exp-C (best prior)",    P_ML3_EXPC["return"],  P_ML3_EXPC["sharpe"],   P_ML3_EXPC["maxdd"]),
    ("P-ML4 (3yr RegimeEnsemble)",  P_ML4["return"],       P_ML4["sharpe"],        P_ML4["maxdd"]),
    ("P-ML5 (6yr RegimeEnsemble)",  m_p5["total_return"],  m_p5["sharpe_ratio"],   m_p5["max_drawdown"]),
]
for name, ret, shr, mdd in rows:
    print(f"  {name:<33} {ret*100:>+8.1f}%  {shr:>+7.3f}  {mdd*100:>7.1f}%")


Equity metrics — P-ML5 vs prior baselines:
Strategy                               Return   Sharpe    MaxDD
---------------------------------------------------------------
  Buy & Hold                          +299.6%   +1.379    -35.4%
  P-ML3 Exp-C (best prior)              +8.8%   +0.280    -49.8%
  P-ML4 (3yr RegimeEnsemble)            -2.5%   +0.227    -57.3%
  P-ML5 (6yr RegimeEnsemble)          +630.2%   +0.927    -68.0%


## §7 — Conclusions / Finding F12

Summary of P-ML5 results and whether extending to 6yr fixed the bull model.

In [12]:
# Collect key results for summary
n_bull_positive = sum(1 for ic in bull_ics_detail
                      if not np.isnan(ic) and ic > 0)
n_bull_fitted   = sum(1 for r in fold_results_P5 if r["bull_used"])
mean_bull_ic    = np.nanmean(bull_ics_detail)
mean_nb_ic      = np.nanmean(nb_ics_detail)
p5_return       = m_p5["total_return"]
p5_sharpe       = m_p5["sharpe_ratio"]
p5_maxdd        = m_p5["max_drawdown"]

beats_p4 = p5_sharpe > P_ML4["sharpe"]
beats_p3 = p5_sharpe > P_ML3_EXPC["sharpe"]
bull_improved = mean_bull_ic > np.nanmean(p4_bull_ics)

print("=" * 70)
print("FINDING F12 — Extended Dataset (6yr) for Regime-Specific Models")
print("=" * 70)
print()
print("Hypothesis: 6yr data gives enough bull bars for the bull model to")
print("learn trend-following (positive IC) vs P-ML4's 3yr negative IC.")
print()
print(f"Bull model folds fitted: {n_bull_fitted}/{N_SPLITS}")
print(f"Bull model IC: Mean={mean_bull_ic:+.4f}  (P-ML4 reference: {np.nanmean(p4_bull_ics):+.4f})")
print(f"Bull IC turned positive: {n_bull_positive}/{n_bull_fitted} folds")
print(f"Non-bull model IC: Mean={mean_nb_ic:+.4f}")
print()
print(f"P-ML5 equity:  Return={p5_return*100:+.1f}%  Sharpe={p5_sharpe:+.3f}  MaxDD={p5_maxdd*100:.1f}%")
print(f"P-ML4 equity:  Return={P_ML4['return']*100:+.1f}%  Sharpe={P_ML4['sharpe']:+.3f}  MaxDD={P_ML4['maxdd']*100:.1f}%")
print(f"P-ML3 Exp-C:   Return={P_ML3_EXPC['return']*100:+.1f}%  Sharpe={P_ML3_EXPC['sharpe']:+.3f}  MaxDD={P_ML3_EXPC['maxdd']*100:.1f}%")
print()
print(f"P-ML5 beats P-ML4? {'YES' if beats_p4 else 'NO'} (Sharpe: {p5_sharpe:+.3f} vs {P_ML4['sharpe']:+.3f})")
print(f"P-ML5 beats P-ML3 Exp-C? {'YES' if beats_p3 else 'NO'} (Sharpe: {p5_sharpe:+.3f} vs {P_ML3_EXPC['sharpe']:+.3f})")
print(f"Bull model IC improved? {'YES' if bull_improved else 'NO'} "
      f"({mean_bull_ic:+.4f} vs {np.nanmean(p4_bull_ics):+.4f})")
print()

# Determine verdict
if n_bull_positive >= 2 and beats_p4:
    verdict = (
        "HYPOTHESIS SUPPORTED: Extending to 6yr data gave the bull model "
        "enough training bars to turn IC positive in multiple folds. "
        "P-ML5 outperforms P-ML4."
    )
elif bull_improved and beats_p4:
    verdict = (
        "HYPOTHESIS PARTIALLY SUPPORTED: Bull model IC improved and P-ML5 beats "
        "P-ML4, but bull IC did not consistently turn positive. "
        "More data helps but does not fully resolve the trend-learning problem."
    )
elif bull_improved:
    verdict = (
        "HYPOTHESIS PARTIALLY SUPPORTED: Bull model IC improved over P-ML4, "
        "but P-ML5 equity does not beat P-ML4. Regime-gating (P-ML3 Exp-C) "
        "remains the best approach."
    )
else:
    verdict = (
        "HYPOTHESIS REJECTED: Bull model IC did not improve despite more data. "
        "1-day horizon trend-following may not be learnable from regime labels alone. "
        "Alternative: longer horizon (3–5d) or momentum feature engineering."
    )

print(f"VERDICT: {verdict}")
print()
print("Next steps:")
print("  - If bull IC still negative: try 3–5d forward return horizon (P-ML6)")
print("  - Add momentum features (ret_mean_20, momentum z-score, quarterly return)")
print("  - Consider HMM-based regime classifier vs rule-based SMA200+ADX")

FINDING F12 — Extended Dataset (6yr) for Regime-Specific Models

Hypothesis: 6yr data gives enough bull bars for the bull model to
learn trend-following (positive IC) vs P-ML4's 3yr negative IC.

Bull model folds fitted: 4/5
Bull model IC: Mean=+0.0212  (P-ML4 reference: -0.1018)
Bull IC turned positive: 3/4 folds
Non-bull model IC: Mean=+0.0773

P-ML5 equity:  Return=+630.2%  Sharpe=+0.927  MaxDD=-68.0%
P-ML4 equity:  Return=-2.5%  Sharpe=+0.227  MaxDD=-57.3%
P-ML3 Exp-C:   Return=+8.8%  Sharpe=+0.280  MaxDD=-49.8%

P-ML5 beats P-ML4? YES (Sharpe: +0.927 vs +0.227)
P-ML5 beats P-ML3 Exp-C? YES (Sharpe: +0.927 vs +0.280)
Bull model IC improved? YES (+0.0212 vs -0.1018)

VERDICT: HYPOTHESIS SUPPORTED: Extending to 6yr data gave the bull model enough training bars to turn IC positive in multiple folds. P-ML5 outperforms P-ML4.

Next steps:
  - If bull IC still negative: try 3–5d forward return horizon (P-ML6)
  - Add momentum features (ret_mean_20, momentum z-score, quarterly return)
  -